# NHS Hospital Database Generation

# Import necessary libraries

In [1]:
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import uuid
import pandas as pd
import random
import string

# Patients Table

### Columns contain several duplicate values, controlled by numpy random and faker name

In [2]:
# Following NHS blood DATA
# O positive: 36%
# O negative: 14%
# A positive: 28%
# A negative: 8%
# B positive: 8%
# B negative: 3%
# AB positive: 2%
# AB negative: 1%

In [ ]:

# set random seeds to maintain randomness
np.random.seed(144)
Faker.seed(144)

# Number of patients
p_n = 2041

# Generate patient first and last names using Faker
patients_first_name = [Faker().first_name() for i in range(p_n)]
patients_last_name = [Faker().last_name() for i in range(p_n)]

# add extra gender for diversity and to reflect the real world population more accurately, based on UK data
genders = ['Male', 'Female', 'Non-binary', 'Other', 'Prefer not to say']
patients_gender = np.random.choice(genders, p_n, p=[0.4, 0.4, 0.07, 0.08, 0.05])

gab = ['Male', 'Female']
patients_gab = np.random.choice(gab, p_n)

# 0 positive is more common, about 40% of the population, based on NHS data
blood_groups = ['O+', 'O-', 'A+', 'A-', 'B+', 'B-', 'AB+', 'AB-']
patients_bg = np.random.choice(blood_groups, p_n, p=[0.36, 0.14, 0.28, 0.08, 0.08, 0.03, 0.02, 0.01])

# Generate patient date of birth between 1950 and 2006
start_date = datetime(1950, 1, 1)
end_date = datetime(2006, 12, 31)
delta_days = (end_date - start_date).days

random_days = np.random.randint(0, delta_days, p_n)

patients_dob = [
    (start_date + timedelta(days=int(day))).strftime('%Y-%m-%d')
    for day in random_days
]

# Generate patient date of registration between 2007 and 2020
reg_start_date = datetime(2007, 1, 1)
reg_end_date = datetime(2020, 12, 31)
reg_delta_days = (reg_end_date - reg_start_date).days

reg_random_days = np.random.randint(0, reg_delta_days, p_n)

patients_date_of_reg = [
    (reg_start_date + timedelta(days=int(day))).strftime('%Y-%m-%d')
    for day in reg_random_days
]

# Generate patient weights between 50 and 110 kg, heights between 147 and 204 cm, and pain levels between 0 and 7.5
patients_weights_kg = np.round(np.random.uniform(50, 110, p_n), 1)
patients_heights_cm = np.round(np.random.uniform(147, 204, p_n), 1) 
patients_pain_level = np.round(np.random.uniform(0, 7.5, p_n), 2)

# Generate patient postcodes in the format 'LU01' to 'LU10'
postcodes = [f'LU{str(i).zfill(2)}' for i in range(1, 11)]
patients_postcode = np.random.choice(postcodes, p_n, p=[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])

In [ ]:
# `generate_nhs_numbers` function to generate valid NHS numbers based on the checksum algorithm used in the UK. The function generates random 9-digit numbers, calculates the checksum, and ensures that the final NHS number is valid.
def generate_nhs_numbers(n):
    weights = np.arange(10, 1, -1)  # 10 to 2
    
    nhs_numbers = []
    
    while len(nhs_numbers) < n:
        # Generate candidate first 9 digits
        digits = np.random.randint(0, 10, size=(n, 9))
        
        # Compute checksum
        total = np.sum(digits * weights, axis=1)
        remainder = total % 11
        check_digit = 11 - remainder
        
        # Handle special cases
        check_digit[check_digit == 11] = 0
        
        # Valid numbers (exclude check_digit == 10)
        valid_mask = check_digit != 10
        
        valid_digits = digits[valid_mask]
        valid_checks = check_digit[valid_mask]
        
        full_numbers = np.hstack(
            (valid_digits, valid_checks.reshape(-1, 1))
        )
        
        nhs_numbers.extend(full_numbers.tolist())
    
    return np.array(nhs_numbers[:n])

In [13]:
nhs_column = generate_nhs_numbers(p_n) 
patients_nhs_number = ["".join(map(str, row)) for row in nhs_column]

In [ ]:
# Create a DataFrame to hold the patient data
patients_df = pd.DataFrame({
    'nhs_number': patients_nhs_number,
    'first_name': patients_first_name,
    'last_name': patients_last_name,
    'gender_at_birth': patients_gab,
    'gender_identity': patients_gender,
    'blood_group': patients_bg,
    'date_of_birth': patients_dob,
    'date_of_registration': patients_date_of_reg,
    'weight_kg': patients_weights_kg,
    'height_cm': patients_heights_cm,
    'pain_level': patients_pain_level,
    'postcode': patients_postcode
})

# Generate email addresses by combining first name, last name, a random number, and a domain. The random number is added to ensure uniqueness and to avoid duplicates in email addresses.
patients_df['email_address'] = (
    patients_df['first_name'].str.strip() +
    patients_df['last_name'].str.strip() +
    np.random.randint(0, 10000, size=len(patients_df)).astype(str) +
    "@gmail.com"
)

patients_df

,nhs_number,first_name,last_name,gender_at_birth,gender_identity,blood_group,date_of_birth,date_of_registration,weight_kg,height_cm,pain_level,postcode,email_address
0,8636369266,Shane,Collier,Male,Male,O+,1970-12-02,2014-09-06,103.4,169.2,6.21,LU10,ShaneCollier1630@gmail.com
1,4142541412,Shawn,Mitchell,Female,Other,A-,1997-10-04,2020-04-30,57.0,167.1,6.27,LU10,ShawnMitchell2707@gmail.com
2,1969601841,Shirley,Walker,Female,Female,A+,1954-08-04,2019-09-14,82.3,160.7,3.97,LU04,ShirleyWalker5868@gmail.com
3,5414729840,Jennifer,Rivera,Male,Male,O+,1993-05-01,2016-08-11,105.8,203.1,3.09,LU01,JenniferRivera1126@gmail.com
4,0360304087,Lisa,Andrews,Male,Female,A+,2001-02-03,2017-08-04,79.8,165.0,3.26,LU10,LisaAndrews785@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2036,4538211710,Kristen,Taylor,Male,Female,A-,2000-06-23,2020-12-06,92.3,195.6,1.91,LU06,KristenTaylor1349@gmail.com
2037,6409512848,Jessica,Morris,Female,Female,A+,1964-08-07,2007-08-09,101.3,186.5,0.23,LU03,JessicaMorris4433@gmail.com
2038,3220001041,Carol,Cummings,Male,Female,O+,1981-05-27,2013-02-14,74.5,188.3,3.89,LU06,CarolCummings9051@gmail.com
2039,2604585200,Joe,Coleman,Male,Non-binary,A+,1990-12-04,2017-06-12,90.1,153.7,0.88,LU10,JoeColeman3288@gmail.com


In [15]:
patients_df.columns

Index(['nhs_number', 'first_name', 'last_name', 'gender_at_birth',
       'gender_identity', 'blood_group', 'date_of_birth',
       'date_of_registration', 'weight_kg', 'height_cm', 'pain_level',
       'postcode', 'email_address'],
      dtype='str')

# Admissions Table

### Columns contain several duplicate values, controlled by numpy random and faker name

In [ ]:
# set random seeds to maintain randomness
np.random.seed(42)
Faker.seed(42)

# Number of rows, number of admission will surpass number of patients
adm_n = 5587

# Generate unique admission IDs using UUID4, which provides a random and unique identifier for each admission. This ensures that each admission record can be uniquely identified in the database.
admissions_id = [str(uuid.uuid4()) for i in range(adm_n)]

# a single patient can be admitted more than once, while some may have never been admitted
adm_patients_nhs_num = np.random.choice(patients_nhs_number, size=adm_n, replace=True)

# Generate random admission dates between January 1, 2021, and November 30, 2024. The admission date is generated by calculating the total number of days between the start and end dates, then randomly selecting a number of days to add to the start date for each admission.
adm_start_date = datetime(2021, 1, 1)
adm_end_date = datetime(2024, 11, 30)
adm_delta_days = (adm_end_date - adm_start_date).days

adm_random_days = np.random.randint(0, adm_delta_days, adm_n)

admissions_date = [
    (adm_start_date + timedelta(days=int(day))).strftime('%Y-%m-%d')
    for day in adm_random_days
]

# Discharge date should be random and can range anywhere from 1-31 days after admission
stay_durations = np.random.randint(1, 31, size=len(admissions_date))

discharge_date = [
    (datetime.strptime(adm, '%Y-%m-%d') + timedelta(days=int(stay))).strftime('%Y-%m-%d')
    for adm, stay in zip(admissions_date, stay_durations)
]

# Emergency admissions are more common, accounting for 36% of all admissions, followed by elective admissions at 49%, and urgent admissions at 15%. This distribution reflects the typical patterns of hospital admissions, where a significant portion of patients require immediate care (emergency), while a larger portion can be scheduled for non-urgent procedures (elective), and a smaller portion falls in between (urgent).
admission_types = ['Emergency', 'Elective', 'Urgent']
admissions_type = np.random.choice(admission_types, adm_n, p=[0.36, 0.49, 0.15])

# The reasons for admission are distributed as follows: Respiratory (16%), Cardiovascular (18%), Gastrointestinal (14%), Neurological (12%), Infection/Constitutional (15%), and Injury/Trauma (25%). This distribution is designed to reflect common reasons for hospital admissions, with a higher proportion of admissions related to injuries and trauma, followed by cardiovascular and respiratory issues.
reasons_for_admission = ['Respiratory', 'Cardiovascular', 'Gastrointestinal', 'Neurological', 'Infection/Constitutional', 'Injury/Trauma']

admissions_reason = np.random.choice(reasons_for_admission, size=adm_n, p=[0.16, 0.18, 0.14, 0.12, 0.15, 0.25])

In [ ]:
# Generate ICD-10 codes for each admission. The ICD-10 codes consist of a letter followed by two digits, and optionally a decimal point and another digit. The letters are chosen from a predefined list, and the digits are generated randomly. There is also a 70% chance that the code will include a decimal subclass. The distribution of letters is based on common ICD-10 code categories, with certain letters being more prevalent than others to reflect real-world data.

icd_letters = ['A', 'B', 'C', 'E', 'I', 'J', 'M', 'S']

def generate_icd10(n):
    chosen_letters = np.random.choice(icd_letters, size=n, p=[0.075, 0.075, 0.12, 0.10, 0.20, 0.16, 0.12, 0.15])
    
    codes = []
    for letter in chosen_letters:
        digits = f"{random.randint(0,99):02d}"
        
        # 70% chance of having decimal subclass
        if random.random() < 0.7:
            decimal = f".{random.randint(0,9)}"
        else:
            decimal = ""
            
        codes.append(letter + digits + decimal)
    
    return codes

admissions_icd_code = generate_icd10(adm_n)

In [ ]:
# Include several duplicates in the physician column to reflect the real world, where certain physicians may have more patients or admissions than others. This also adds realism to the dataset by simulating a typical hospital environment where some doctors are more frequently involved in patient care than others.
admission_attending_physicians = [Faker().name() for _ in range(186)]
admissions_attending_physician = np.random.choice(admission_attending_physicians, size=adm_n)

In [20]:
# Ward + Bay + Bed (Very Realistic NHS Format)

# UK hospitals often use:
# Ward-Bay-Bed

def generate_room_numbers(n):
    wards = np.random.choice(list("ABCDEFGH"), size=n)
    bays = np.random.randint(1, 6, size=n)   # 1–5 bays
    beds = np.random.randint(1, 7, size=n)   # 1–6 beds per bay
    
    return [
        f"{ward}-{bay:02d}-{bed}"
        for ward, bay, bed in zip(wards, bays, beds)
    ]

admissions_room_number = generate_room_numbers(adm_n)

In [ ]:
# Generate admission statuses with a realistic distribution. 
admission_statuses = ['Discharged Home', 'Discharged Death', 'Absconded', 'Cancelled', 'Discharged to Facility/Step-down', 'Patient Did Not Attend (DNA)']
admissions_status = np.random.choice(admission_statuses, size=adm_n, p=[0.5, 0.05, 0.03, 0.2, 0.12, 0.1])

In [ ]:
# Create a DataFrame to hold the admissions data, linking it to the patients via the NHS number. 
admissions_df = pd.DataFrame({
    'id': admissions_id,
    'patient_nhs_number': adm_patients_nhs_num,
    'admission_date': admissions_date,
    'discharge_date': discharge_date,
    'type': admissions_type,
    'reason': admissions_reason,
    'icd code': admissions_icd_code,
    'attending physician': admissions_attending_physician,
    'room number': admissions_room_number,
    'status': admissions_status
})

admissions_df.head()

,id,patient_nhs_number,admission_date,discharge_date,type,reason,icd code,attending physician,room number,status
0,9e18c705-428c-4839-a4fc-cda85e3ef38f,9085026997,2024-03-10,2024-03-12,Elective,Respiratory,S71.7,Michael Elliott,B-05-2,Discharged Home
1,7965bb75-4997-4b24-af07-513135b2af16,8037283259,2021-12-28,2022-01-04,Emergency,Infection/Constitutional,I41.1,Michael Wang,G-02-3,Discharged Death
2,48fc04b7-6ac9-4499-a2d6-1da0174c4204,2843187699,2021-01-08,2021-02-06,Elective,Injury/Trauma,S54.9,Sandra Aguilar,F-05-4,Discharged to Facility/Step-down
3,08857a2d-1cdb-49b2-9ff8-36387521c774,6345232521,2022-04-23,2022-05-03,Emergency,Neurological,M62.5,Debra Davidson,G-02-6,Discharged Home
4,1176422a-c2f6-4333-9f0f-2fdb9087e521,8427316593,2024-10-07,2024-10-29,Elective,Cardiovascular,I52.0,David Wright,E-01-5,Discharged Home


# Treatment Table

### Columns contain several duplicate values, controlled by numpy random

In [ ]:
# set random seeds to maintain randomness
np.random.seed(42)
Faker.seed(42)

# Number of rows, number of admission will surpass number of patients
t_n = 9952

# Generate unique treatment IDs using UUID4, which provides a random and unique identifier for each treatment. This ensures that each treatment record can be uniquely identified in the database.
treatments_id = [str(uuid.uuid4()) for i in range(t_n)]

# a single patient can be admitted more than once, while some may have never been admitted
treatments_adm_id = np.random.choice(admissions_id, size=t_n, replace=True)

# Generate treatment types with a realistic distribution
treatment_types = ['Medications', 'Diagnostic Tests', 'Imaging', 'Surgical Procedures', 'Therapies']
treatments_type = np.random.choice(treatment_types, size=t_n, p=[0.3, 0.3, 0.15, 0.15, 0.1])

# Generate treatment dates that fall between the corresponding admission and discharge dates. 
t_start_date = datetime(2021, 1, 3)
t_end_date = datetime(2024, 12, 15)
t_delta_days = (t_end_date - t_start_date).days

t_random_days = np.random.randint(0, t_delta_days, t_n)

treatments_date = [
    (t_start_date + timedelta(days=int(day))).strftime('%Y-%m-%d')
    for day in t_random_days
]

# Generate treatment costs in pounds, ranging from £10 to £399, to reflect the wide range of possible treatment expenses in a hospital setting. 
treatment_cost_pounds = np.random.randint(10, 399, t_n)



In [ ]:
# Create a DataFrame to hold the treatments data, linking it to the admissions via the admission ID.
t_df = pd.DataFrame({
    'id': treatments_id,
    'admission_id': treatments_adm_id,
    'type': treatments_type,
    'date': treatments_date,
    'cost (£)': treatment_cost_pounds
})

t_df

,id,admission_id,type,date,cost (£)
0,58f62318-103b-43be-99c4-00e8ec451263,5a6eedca-57e4-4ce2-aa18-8b3d9315747a,Therapies,2024-05-20,16
1,7ff69e35-31e2-4ee6-98c3-d6903f815a79,fbc718ef-52df-44d6-b68f-d8ab4e86555a,Surgical Procedures,2022-11-03,215
2,b19271cd-4a18-4ec5-8ef1-bc9c9c546e55,2935ed36-dca2-474b-965d-7373b434a759,Medications,2024-12-10,188
3,4cd42593-eb02-482f-b3e5-9d26e223a520,4e57f7c2-33cb-4baf-965f-195cbc272825,Surgical Procedures,2022-10-29,192
4,02ee11e3-a23e-4d49-b006-ee77360c704f,b4678c2a-9ce6-4ad1-bcb3-84b37304fd1e,Surgical Procedures,2023-05-08,237
...,...,...,...,...,...
9947,6537ec75-d8da-499c-b6a5-231304810708,7a155498-ce4a-4daf-8821-b7bcb30b2292,Imaging,2024-08-25,253
9948,a57bf15f-1d13-42af-bc63-578c62e937ca,d9135c1c-9f47-4eaa-8701-a2da24bb683b,Medications,2023-02-21,330
9949,a483cd20-9624-4054-8a42-e4827ffb8a6e,244b6268-2f3a-492e-95b8-6151c3c501da,Medications,2023-08-03,140
9950,966e997f-85da-4336-8108-c8c84ad86131,a9ff9aa7-d610-4ffd-b8bc-44b33a138870,Diagnostic Tests,2024-02-26,361


# Add nan values to specific columns in the tables

In [ ]:
def add_column_specific_nans(df, column_nan_rates, random_state=None):
    """
    column_nan_rates: dict like {'height_cm': 0.1, 'weight_kg': 0.2}
    """
    df_copy = df.copy()
    np.random.seed(random_state)
    
    for col, rate in column_nan_rates.items():
        n_nans = int(len(df_copy) * rate)
        nan_indices = np.random.choice(df_copy.index, n_nans, replace=False)
        df_copy.loc[nan_indices, col] = np.nan
        
    return df_copy

In [26]:
patients_df = add_column_specific_nans(
    patients_df,
    column_nan_rates={
        'height_cm': 0.0535,
        'email_address': 0.01374,
        'weight_kg': 0.01864,
        'pain_level': 0.01379,
        'postcode': 0.074376
    },
    random_state=42
)

admissions_df = add_column_specific_nans(
    admissions_df,
    column_nan_rates={
        'room number': 0.06473,
        'status': 0.01857,
        'reason': 0.01647,
        'attending physician': 0.018,
        'type': 0.01836
    },
    random_state=42
)

t_df = add_column_specific_nans(
    t_df,
    column_nan_rates={
        'cost (£)': 0.01473,
        'date': 0.01638,
        'type': 0.036795,
    },
    random_state=42
)

In [27]:
t_df

,id,admission_id,type,date,cost (£)
0,58f62318-103b-43be-99c4-00e8ec451263,5a6eedca-57e4-4ce2-aa18-8b3d9315747a,Therapies,2024-05-20,16.0
1,7ff69e35-31e2-4ee6-98c3-d6903f815a79,fbc718ef-52df-44d6-b68f-d8ab4e86555a,Surgical Procedures,2022-11-03,215.0
2,b19271cd-4a18-4ec5-8ef1-bc9c9c546e55,2935ed36-dca2-474b-965d-7373b434a759,Medications,2024-12-10,188.0
3,4cd42593-eb02-482f-b3e5-9d26e223a520,4e57f7c2-33cb-4baf-965f-195cbc272825,Surgical Procedures,2022-10-29,192.0
4,02ee11e3-a23e-4d49-b006-ee77360c704f,b4678c2a-9ce6-4ad1-bcb3-84b37304fd1e,Surgical Procedures,2023-05-08,237.0
...,...,...,...,...,...
9947,6537ec75-d8da-499c-b6a5-231304810708,7a155498-ce4a-4daf-8821-b7bcb30b2292,Imaging,2024-08-25,253.0
9948,a57bf15f-1d13-42af-bc63-578c62e937ca,d9135c1c-9f47-4eaa-8701-a2da24bb683b,Medications,2023-02-21,330.0
9949,a483cd20-9624-4054-8a42-e4827ffb8a6e,244b6268-2f3a-492e-95b8-6151c3c501da,Medications,2023-08-03,140.0
9950,966e997f-85da-4336-8108-c8c84ad86131,a9ff9aa7-d610-4ffd-b8bc-44b33a138870,Diagnostic Tests,2024-02-26,361.0


# Output all the tables to a CSV file

In [28]:
patients_df.to_csv('patients.csv', index=False)
admissions_df.to_csv('admissions.csv', index=False)
t_df.to_csv('treatments.csv', index=False)